# Annual employment data preparation

This notebook combines the Ireland-level annual employment rate with the annual percentage of part-time work, then derives part-time and full-time employment rates.

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Load the two source tables

In [19]:
data_dir = Path('.')
employment = pd.read_csv(data_dir / 'AnnualEmploymentRate.csv')
part_time = pd.read_csv(data_dir / 'AnnualPercentageOfPart-timeWork.csv')

print(f'Employment rows: {len(employment):,}')
print(f'Part-time rows: {len(part_time):,}')
display(employment.head())
display(part_time.head())

Employment rows: 2,838
Part-time rows: 576


,Year,Age Group,Sex,Education Level,Region,Employment Rate
0,2019,All ages,Both sexes,All levels,Ireland,74.9
1,2019,All ages,Both sexes,All levels,Northern and Western,74.0
2,2019,All ages,Both sexes,All levels,Southern,72.8
3,2019,All ages,Both sexes,All levels,Eastern and Midland,76.6
4,2019,All ages,Both sexes,Primary / Lower Secondary (1–2),Ireland,52.8


,Year,Age Group,Sex,Education Level,Part-time Work Percentage
0,2019,All ages,Both sexes,All levels,18.1
1,2019,All ages,Both sexes,Primary / Lower Secondary (1–2),25.1
2,2019,All ages,Both sexes,Upper / Post-Secondary (3–4),22.9
3,2019,All ages,Both sexes,Tertiary (5-8),13.3
4,2019,All ages,Male,All levels,8.7


## 2. Keep the all-region employment records

`Ireland` is the all-region value in `AnnualEmploymentRate.csv`. After filtering, `Region` is no longer needed.

In [20]:
employment_all_region = (
    employment.loc[employment['Region'].eq('Ireland')]
    .drop(columns='Region')
    .copy()
)

print(f'All-region employment rows: {len(employment_all_region):,}')
display(employment_all_region.head())

All-region employment rows: 771


,Year,Age Group,Sex,Education Level,Employment Rate
0,2019,All ages,Both sexes,All levels,74.9
4,2019,All ages,Both sexes,Primary / Lower Secondary (1–2),52.8
8,2019,All ages,Both sexes,Upper / Post-Secondary (3–4),72.3
12,2019,All ages,Both sexes,Tertiary (5-8),84.9
16,2019,All ages,Male,All levels,81.3


## 3. Merge the percentage of part-time work

The four shared dimensions form the merge key. A left, one-to-one merge retains every Ireland-level employment record so that unavailable part-time percentages can be investigated explicitly.

In [21]:
merge_keys = ['Year', 'Age Group', 'Sex', 'Education Level']

assert not employment_all_region.duplicated(merge_keys).any(), 'Duplicate employment merge keys found.'
assert not part_time.duplicated(merge_keys).any(), 'Duplicate part-time merge keys found.'

combined = employment_all_region.merge(
    part_time,
    on=merge_keys,
    how='left',
    validate='one_to_one',
)

print(f'Combined rows: {len(combined):,}')
print(f'Matched part-time percentages: {combined["Part-time Work Percentage"].notna().sum():,}')
display(combined.head())

Combined rows: 771
Matched part-time percentages: 576


,Year,Age Group,Sex,Education Level,Employment Rate,Part-time Work Percentage
0,2019,All ages,Both sexes,All levels,74.9,18.1
1,2019,All ages,Both sexes,Primary / Lower Secondary (1–2),52.8,25.1
2,2019,All ages,Both sexes,Upper / Post-Secondary (3–4),72.3,22.9
3,2019,All ages,Both sexes,Tertiary (5-8),84.9,13.3
4,2019,All ages,Male,All levels,81.3,8.7


## 4. Calculate part-time and full-time employment rates

Both source measures are stored as percentages from 0 to 100, so `Part-time Work Percentage` is divided by 100 before multiplication.

In [22]:
part_time_share = combined['Part-time Work Percentage'] / 100

combined['Part-time Employment Rate'] = (
    combined['Employment Rate'] * part_time_share
).round(2)

combined['Full-time Employment Rate'] = (
    combined['Employment Rate'] * (1 - part_time_share)
).round(2)

has_part_time_percentage = combined['Part-time Work Percentage'].notna()
assert np.allclose(
    combined.loc[has_part_time_percentage, 'Part-time Employment Rate']
    + combined.loc[has_part_time_percentage, 'Full-time Employment Rate'],
    combined.loc[has_part_time_percentage, 'Employment Rate'],
    atol=0.011,
), 'Derived rates do not add up to the employment rate.'

display(combined.head())

,Year,Age Group,Sex,Education Level,Employment Rate,Part-time Work Percentage,Part-time Employment Rate,Full-time Employment Rate
0,2019,All ages,Both sexes,All levels,74.9,18.1,13.56,61.34
1,2019,All ages,Both sexes,Primary / Lower Secondary (1–2),52.8,25.1,13.25,39.55
2,2019,All ages,Both sexes,Upper / Post-Secondary (3–4),72.3,22.9,16.56,55.74
3,2019,All ages,Both sexes,Tertiary (5-8),84.9,13.3,11.29,73.61
4,2019,All ages,Male,All levels,81.3,8.7,7.07,74.23


## 5. Add a part-time data availability field

`Has Part-time Data` is `True` when `Part-time Work Percentage` is available and `False` when it is missing. This field can be used directly by interactive visualisation filters. The detailed table and summary below also investigate the missing records.

In [23]:
combined['Has Part-time Data'] = combined['Part-time Work Percentage'].notna()

print(combined['Has Part-time Data'].value_counts(dropna=False))

missing_part_time = combined.loc[
    ~combined['Has Part-time Data'],
    merge_keys + ['Employment Rate'],
].copy()

missing_count = len(missing_part_time)
missing_rate = missing_count / len(combined) * 100
print(f'Missing Part-time Work Percentage: {missing_count:,} of {len(combined):,} rows ({missing_rate:.2f}%)')
display(missing_part_time)

missing_summary_parts = []
for dimension in merge_keys:
    dimension_counts = (
        missing_part_time[dimension]
        .value_counts(dropna=False)
        .rename_axis('Category')
        .reset_index(name='Missing Rows')
    )
    dimension_counts.insert(0, 'Dimension', dimension)
    missing_summary_parts.append(dimension_counts)

missing_summary = pd.concat(missing_summary_parts, ignore_index=True)
display(missing_summary)

Has Part-time Data
True     576
False    195
Name: count, dtype: int64
Missing Part-time Work Percentage: 195 of 771 rows (25.29%)


,Year,Age Group,Sex,Education Level,Employment Rate
17,2019,20 - 24,Male,Primary / Lower Secondary (1–2),32.4
21,2019,20 - 24,Female,Primary / Lower Secondary (1–2),25.9
29,2019,25 - 29,Male,Primary / Lower Secondary (1–2),55.7
33,2019,25 - 29,Female,Primary / Lower Secondary (1–2),29.4
53,2019,35 - 39,Male,Primary / Lower Secondary (1–2),67.6
...,...,...,...,...,...
754,2025,55 - 59,Male,Tertiary (5-8),91.0
756,2025,55 - 59,Female,Primary / Lower Secondary (1–2),42.2
764,2025,60 - 64,Male,Primary / Lower Secondary (1–2),65.8
765,2025,60 - 64,Male,Upper / Post-Secondary (3–4),71.6


,Dimension,Category,Missing Rows
0,Year,2022,38
1,Year,2024,37
2,Year,2021,36
3,Year,2025,36
4,Year,2023,33
5,Year,2020,10
6,Year,2019,5
7,Age Group,25 - 29,28
8,Age Group,30 - 34,27
9,Age Group,40 - 44,25


## 6. Save the combined table

In [24]:
combined = combined.sort_values(merge_keys).reset_index(drop=True)
output_path = data_dir / 'AnnualEmploymentByWorkTime.csv'
combined.to_csv(output_path, index=False)

print(f'Saved {len(combined):,} rows to {output_path}')
print('Columns:')
for column in combined.columns:
    print(f'  - {column}')

Saved 771 rows to AnnualEmploymentByWorkTime.csv
Columns:
  - Year
  - Age Group
  - Sex
  - Education Level
  - Employment Rate
  - Part-time Work Percentage
  - Part-time Employment Rate
  - Full-time Employment Rate
  - Has Part-time Data
